In [0]:
%run ../dim_full_load/Configuration

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
source_table = "dbo.FACT_TRANSACTION"
bronze_table = "learning_lab_catalog.bronze.fact_transaction"

In [0]:
last_ts = spark.sql(f"""
    SELECT
        COALESCE(
            MAX(try_to_timestamp(updated_at)),
            TIMESTAMP '1900-01-01 00:00:00'
        ) AS last_ts
    FROM {bronze_table}
""").collect()[0][0]

print("Last processed timestamp:", last_ts)


df_bronze_fact = spark.read.jdbc(
    url=jdbc_url,
    table=f"""
        (
            SELECT *
            FROM {source_table}
            WHERE updated_at >
                  CAST('{last_ts.strftime("%Y-%m-%d %H:%M:%S")}' AS datetime2)
        ) t
    """,
    properties=jdbc_properties
)


In [0]:
df_bronze_fact.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(bronze_table)

spark.sql(f"""
    UPDATE {bronze_table}
    SET updated_at = NULL
    WHERE try_to_timestamp(updated_at) IS NULL
""")

In [0]:
%sql
SELECT count(*) 
FROM learning_lab_catalog.bronze.fact_transaction
WHERE updated_at > '2026-01-13 08:03:48'